<a href="https://colab.research.google.com/github/Zahra-Mhdi/Deep-Learning-Exercises/blob/main/Session_7_Sequence_Models_Exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Cell 1 — Setup and Imports

In [1]:
import math
import time
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import requests
import re

device = "cuda" if torch.cuda.is_available() else "cpu"
device


'cpu'

Cell 2 — Load and Clean Dataset

In [2]:
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text = requests.get(url).text
text = text.replace("\r\n", "\n").replace("\r", "\n")
text = "\n".join(line.rstrip() for line in text.split("\n"))
text = re.sub(r"[ ]{2,}", " ", text)
text = re.sub(r"\n{3,}", "\n\n", text)
len(text)


1115374

Cell 3 — Tokenizer

In [3]:
chars = sorted(list(set(text)))
vocab = {ch:i for i,ch in enumerate(chars)}
ivocab = {i:ch for ch,i in vocab.items()}
vocab_size = len(vocab)

def encode(s):
    return [vocab[ch] for ch in s]

def decode(ids):
    return "".join(ivocab[i] for i in ids)


Cell 4 — Train and Validation Split

In [4]:
encoded = encode(text)
n = len(encoded)
train_ids = encoded[:int(0.9*n)]
val_ids = encoded[int(0.9*n):int(0.95*n)]


Cell 5 — Dataset and DataLoader

In [5]:
class CharDataset(Dataset):
    def __init__(self, ids, seq_len=128):
        self.data = torch.tensor(ids, dtype=torch.long)
        self.seq_len = seq_len

    def __len__(self):
        return len(self.data) - self.seq_len

    def __getitem__(self, idx):
        x = self.data[idx:idx+self.seq_len]
        y = self.data[idx+1:idx+self.seq_len+1]
        return x, y

seq_len = 128
batch_size = 64

train_ds = CharDataset(train_ids, seq_len=seq_len)
val_ds = CharDataset(val_ids, seq_len=seq_len)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, drop_last=True)


Cell 6 — Model (RNN/GRU/LSTM)

In [6]:
class CharRNN(nn.Module):
    def __init__(self, vocab_size, embed_size=128, hidden_size=256, num_layers=2, rnn_type="lstm"):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_size)
        self.rnn_type = rnn_type.lower()
        if self.rnn_type == "rnn":
            self.rnn = nn.RNN(embed_size, hidden_size, num_layers=num_layers, batch_first=True)
        elif self.rnn_type == "gru":
            self.rnn = nn.GRU(embed_size, hidden_size, num_layers=num_layers, batch_first=True)
        else:
            self.rnn = nn.LSTM(embed_size, hidden_size, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden=None):
        x = self.embed(x)
        out, hidden = self.rnn(x, hidden)
        logits = self.fc(out)
        return logits, hidden


Cell 7 — Hidden Detach

In [7]:
def detach_hidden(hidden):
    if hidden is None:
        return None
    if isinstance(hidden, tuple):
        return (hidden[0].detach(), hidden[1].detach())
    return hidden.detach()


Cell 8 — Train Epoch

In [8]:
criterion = nn.CrossEntropyLoss()

def train_epoch(model, loader, optimizer, clip=1.0, max_batches=400):
    model.train()
    total = 0.0
    hidden = None
    c = 0
    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)
        optimizer.zero_grad()
        logits, hidden = model(xb, hidden)
        hidden = detach_hidden(hidden)
        loss = criterion(logits.view(-1, logits.size(-1)), yb.view(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        total += loss.item()
        c += 1
        if c >= max_batches:
            break
    return total / c



Cell 9 — Validation Epoch

In [9]:
@torch.no_grad()
def eval_epoch(model, loader, max_batches=200):
    model.eval()
    total = 0.0
    hidden = None
    c = 0
    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)
        logits, hidden = model(xb, hidden)
        hidden = detach_hidden(hidden)
        loss = criterion(logits.view(-1, logits.size(-1)), yb.view(-1))
        total += loss.item()
        c += 1
        if c >= max_batches:
            break
    avg = total / c
    return avg, math.exp(avg)


Cell 10 — Optimizer Builder

In [10]:
def make_optimizer(name, params, lr):
    name = name.lower()
    if name == "adamw":
        return torch.optim.AdamW(params, lr=lr)
    if name == "sgd":
        return torch.optim.SGD(params, lr=lr, momentum=0.9)
    return torch.optim.Adam(params, lr=lr)


Cell 11 — Training Experiments

In [11]:
exp_results = []

lrs = [3e-4, 1e-3]
clips = [1.0]
opts = ["adam", "adamw"]

for lr in lrs:
    for clip in clips:
        for opt_name in opts:
            model = CharRNN(vocab_size, 128, 256, 2, "lstm").to(device)
            optimizer = make_optimizer(opt_name, model.parameters(), lr)
            train_epoch(model, train_loader, optimizer, clip=clip, max_batches=80)
            vloss, vppl = eval_epoch(model, val_loader, max_batches=40)
            exp_results.append((lr, clip, opt_name, vloss, vppl))

exp_results


[(0.0003, 1.0, 'adam', 3.0686728835105894, 21.513333022570727),
 (0.0003, 1.0, 'adamw', 3.0803548991680145, 21.766125805539524),
 (0.001, 1.0, 'adam', 2.445442807674408, 11.535656545305457),
 (0.001, 1.0, 'adamw', 2.4637756526470183, 11.749088377033925)]

Cell 12 — Generation

In [12]:
@torch.no_grad()
def generate(model, start_string="To", length=300, temperature=1.0):
    model.eval()
    ids = encode(start_string)
    h = None
    x = torch.tensor([[ids[0]]], dtype=torch.long).to(device)

    for i in range(1, len(ids)):
        _, h = model(x, h)
        x = torch.tensor([[ids[i]]], dtype=torch.long).to(device)

    out = list(start_string)

    for _ in range(length):
        logits, h = model(x, h)
        logits = logits[:, -1, :] / max(1e-8, float(temperature))
        probs = torch.softmax(logits, dim=-1)
        idx = torch.multinomial(probs, 1).item()
        out.append(ivocab[idx])
        x = torch.tensor([[idx]], dtype=torch.long).to(device)

    return "".join(out)


Cell 13 — Final LSTM Training (2 Epochs)

In [13]:
model = CharRNN(vocab_size, 128, 256, 2, "lstm").to(device)
optimizer = make_optimizer("adam", model.parameters(), 1e-3)

for epoch in range(1, 3):
    tloss = train_epoch(model, train_loader, optimizer, clip=1.0, max_batches=300)
    vloss, vppl = eval_epoch(model, val_loader, max_batches=200)
    print(epoch, tloss, vloss, vppl)

print(generate(model, "First Citizen:\n", 300, 0.7))
print("----")
print(generate(model, "First Citizen:\n", 300, 1.0))
print("----")
print(generate(model, "First Citizen:\n", 300, 1.3))


1 2.2914293173948925 1.8727731859683991 6.506314624216785
2 1.6670151233673096 1.6565541857481003 5.241219423922182
First Citizen:
And suld nother, I will me to senot as say send in betiend
And never 'buse with honour's given of were at all that is
Good than part the roght to grok me of the shall
the by thou not your fair: since, humble enough
but consence, the counter drown:
Who then he be would not aly storning. A aid,
And we
----
First Citizen:
Then, no make; , at timons unless nones,
And staks of his, you he have thee dugnt to beer
Myseffere an eyet 'tis saxe:
The torks for?

WARWICHS:
Mrant you knee is the your how behedia as fulss it so
beger.
What had forse:
That I mace Yord.

FRIAR LAURENCE:
Waw, you, my July, for foul up,
To shall fe
----
First Citizen:
To morticual dumque. Beshorn my faths! What? Edwards, what whirs in one fall
Rawing itfollwited eand?
'Giken an the senition!

DUCHESS OF YORK:
That eye weightex-lords;
Hown reloveved me!

JORFLY:
No wrongs thit dlbsh man, eocr

Cell 14 — Ablation Study

In [14]:
ablation = []

for rnn_type in ["rnn", "gru", "lstm"]:
    for hidden in [128, 256]:
        model = CharRNN(vocab_size, 128, hidden, 2, rnn_type).to(device)
        optimizer = make_optimizer("adam", model.parameters(), 1e-3)
        train_epoch(model, train_loader, optimizer, clip=1.0, max_batches=150)
        vloss, vppl = eval_epoch(model, val_loader, max_batches=100)
        s07 = generate(model, "First Citizen:\n", 200, 0.7)
        s10 = generate(model, "First Citizen:\n", 200, 1.0)
        s13 = generate(model, "First Citizen:\n", 200, 1.3)
        ablation.append((rnn_type, hidden, vloss, vppl, s07, s10, s13))

ablation


[('rnn',
  128,
  2.080423895120621,
  8.007862688863705,
  'First Citizen:\nAnd hing make and the herilcon the surfet me you that I wilh To hout and the poow the the pouferse, the care mare procher the hese stece thoust the her his my un to the the soor hand has to ee buised ',
  "First Citizen:\nFarther to sod-\nYou show.\n\nOLIARNO:\n\nPUCICINGBGOY undemire wisung eavend'd mary nolet tow mone dobe.\nA thove theas the me arr Loved the one ladeu: to to I brow thifnd of cack. el\nWork. qhe nounfor my ",
  "First Citizen:\nhe.\nSuld.\nA pnach Ares; my tha, dav; pradonith devcotide dearl!\nAh to sy midry.\n\nBur?;\nPnow, hef tnout,bis Burobt stlone..\nFtel you, Agail hom m'se\nCovasti.\nH chike,\ntry kmawhroupponge-\nGoaks eat:\nFin"),
 ('rnn',
  256,
  1.9503987276554107,
  7.031490671506713,
  'First Citizen:\nWhere in with that hade I lefe truthen,\nThe with ther thes, to then more a stack.\n\nCOMENCERHART:\nWhing of mand\nCangarmone and may tall dest st finsomenowh so me to prather t